# Forbes America's Top Colleges — Exploratory Data Analysis

### A Guided Practice Notebook (Questions Only)

This notebook walks through a real-world exploratory data analysis (EDA) exercise using the
**Forbes America's Top Colleges** dataset. Each section below poses a question, explains the
underlying pandas concept, gives you a hint, and describes what your output should look like.
Your task is to write the code yourself in the empty cell provided under each question.

---

## Context

Since 2008, Forbes Magazine has published an annual ranking of America's best colleges. For a
question as important as *"Is college worth it?"*, this list is a widely cited reference point —
helping prospective students weigh academics, campus experience, career outcomes, and student debt
before choosing where to study.

The Forbes ranking methodology reviews undergraduate institutions across several dimensions:
academic quality, student experience, post-graduation career success, and affordability
(student debt levels).

## About the Data

The dataset contains rankings and statistics for **650 United States colleges**, including fields
such as rank, name, location (state/city), public or private status, student population,
undergraduate population, acceptance rate, average grant aid, and average alumni salary.

## Learning Objectives

By the end of this notebook, you should be comfortable with:

- Loading and inspecting a dataset with pandas (`shape`, `info`, `describe`)
- Detecting and handling missing values
- Cleaning up column names and resetting indexes
- Filtering rows with boolean conditions
- Grouping and aggregating data with `groupby`
- Sorting values to find "top N" style answers
- Creating new calculated columns
- Searching text columns with string methods
- Binning a continuous variable into categories with `pd.cut`
- Selecting data by position and label using `iloc` and `loc`

---


## Setup

### Load Required Packages

We will use **pandas** for data manipulation and **numpy** for numerical operations. These are the
two core libraries for almost any data analysis task in Python.


In [5]:
import os
import pandas as pd
import numpy as np

os.getcwd()


'/kaggle/working'

### Load the Dataset

Make sure the file `AmericasTopColleges.csv` is saved in the same folder as this notebook
(or update the path below) before running the cell.


In [6]:
ac = pd.read_csv("/kaggle/input/datasets/rishab023/american-dataset/AmericasTopColleges.csv")


---

## Warm-Up Exercises

Before jumping into analysis questions, it's good practice to always inspect a new dataset first:
how big is it, what type of data does each column hold, and is anything missing or messy? The
exercises below build that habit.


### Warm-Up 1: Check the Dimensions of the Data

**Concept:** The `.shape` attribute of a DataFrame returns a tuple `(number_of_rows, number_of_columns)`.
It's usually the very first thing you check after loading any dataset, so you know the scale of
what you're working with.

**Hint:** Simply call the `.shape` attribute on your DataFrame — no parentheses needed, since it's
an attribute, not a method.

**Expected Output:** A tuple of two integers, e.g. `(650, N)`, representing rows and columns.


In [7]:
ac.shape

(650, 17)

### Warm-Up 2: Check the Data Types of Each Field

**Concept:** The `.info()` method summarizes the DataFrame: column names, non-null counts, and data
types (`int64`, `float64`, `object`, etc.). This tells you which columns are numeric and which are
text — important before you try to do math on a column.

**Hint:** Call `.info()` on your DataFrame.

**Expected Output:** A printed summary listing every column name, its non-null count, and its dtype,
plus total memory usage.


In [8]:
ac.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 650 entries, 0 to 649
Data columns (total 17 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Rank                      650 non-null    int64  
 1   Name                      650 non-null    object 
 2   City                      650 non-null    object 
 3   State                     650 non-null    object 
 4   Public/Private            650 non-null    object 
 5   Undergraduate Population  650 non-null    int64  
 6   Student Population        650 non-null    int64  
 7   Net Price                 648 non-null    float64
 8   Average Grant Aid         646 non-null    float64
 9   Total Annual Cost         650 non-null    int64  
 10  Alumni Salary             635 non-null    float64
 11  Acceptance Rate           648 non-null    float64
 12  SAT Lower                 551 non-null    float64
 13  SAT Upper                 551 non-null    float64
 14  ACT Lower 

### Warm-Up 3: Check and Handle Missing Values

**Concept:** `.isnull()` returns a same-shaped DataFrame of `True`/`False` values marking where data
is missing. Chaining `.sum()` after it adds up the `True` values per column, giving you a missing-value
count per column. Once you know where the gaps are, a common cleanup approach is to fill missing
**text** columns with the most frequent value (`mode`) and missing **numeric** columns with the
`median` (median is preferred over mean here since it's less sensitive to outliers).

**Hint:** First run `ac.isnull().sum()` to see the gaps. Then loop through each column, checking
`ac[column].dtype`, and use `.fillna()` with `.mode()[0]` for object/text columns or `.median()` for
numeric columns. Remember to pass `inplace=True` (or reassign the column) so the change is saved.

**Expected Output:** First, a Series showing missing-value counts per column. After cleanup, running
`ac.isnull().sum()` again should show all zeros.


In [9]:
ac.isnull().sum()

Rank                         0
Name                         0
City                         0
State                        0
Public/Private               0
Undergraduate Population     0
Student Population           0
Net Price                    2
Average Grant Aid            4
Total Annual Cost            0
Alumni Salary               15
Acceptance Rate              2
SAT Lower                   99
SAT Upper                   99
ACT Lower                   97
ACT Upper                   97
Website                      5
dtype: int64

### Warm-Up 4: Preview the First and Last Rows

**Concept:** `.head()` and `.tail()` show the first/last 5 rows of a DataFrame by default (you can
pass a number, e.g. `.head(10)`, for more). This is a quick sanity check to visually confirm the data
loaded correctly and looks as expected.

**Hint:** Call `.head()` and `.tail()` separately on your DataFrame.

**Expected Output:** Two small tables, each with 5 rows and all columns, showing the beginning and end
of the dataset.


In [10]:
ac.head()
ac.tail()

,Rank,Name,City,State,Public/Private,Undergraduate Population,Student Population,Net Price,Average Grant Aid,Total Annual Cost,Alumni Salary,Acceptance Rate,SAT Lower,SAT Upper,ACT Lower,ACT Upper,Website
645,646,New Mexico State University,Las Cruces,NM,Public,13379,14432,8625.0,9582.0,34720,96700.0,64.0,910.0,1160.0,18.0,23.0,www.nmsu.edu
646,647,Indiana State University,Terre Haute,IN,Public,13626,13763,13012.0,9297.0,32938,85600.0,85.0,900.0,1110.0,17.0,23.0,www.indstate.edu
647,648,Emory &amp; Henry College,Emory,VA,Private,1094,1226,19340.0,27155.0,48100,70700.0,72.0,988.0,1170.0,19.0,25.0,www.ehc.edu
648,649,Wells College,Aurora,NY,Private,488,516,22828.0,30207.0,55180,NaN,80.0,NaN,NaN,NaN,NaN,www.wells.edu
649,650,Catawba College,Salisbury,NC,Private,1336,1463,19807.0,23662.0,44921,78100.0,42.0,930.0,1100.0,18.0,23.0,www.catawba.edu


### Warm-Up 5: Clean Up Column Names

**Concept:** Column names with spaces or special characters (like `/`) are inconvenient to reference
in code, since `ac.Average Grant Aid` would break — you'd need `ac["Average Grant Aid"]` instead.
Standardizing column names (e.g., replacing spaces and slashes with underscores) lets you use the
cleaner dot notation (`ac.Average_Grant_Aid`) throughout the rest of the analysis.

**Hint:** Use `ac.columns.str.replace()` with a pattern matching spaces and slashes, replacing them
with an underscore, and reassign the result back to `ac.columns`.

**Expected Output:** Running `ac.columns` afterward should show column names with underscores instead
of spaces or slashes (e.g., `Public_Private` instead of `Public/Private`).


In [11]:
ac.columns = ac.columns.str.replace(r"[ /]+", "_", regex=True)

### Warm-Up 6: Get a Statistical Summary

**Concept:** `.describe()` computes summary statistics (count, mean, standard deviation, min, max,
and quartiles) for all numeric columns in one call — a fast way to spot outliers or unusual ranges
before deeper analysis.

**Hint:** Call `.describe()` on your DataFrame.

**Expected Output:** A table with statistics (count, mean, std, min, 25%, 50%, 75%, max) as rows, and
each numeric column as a column.


In [12]:
ac.describe()

,Rank,Undergraduate_Population,Student_Population,Net_Price,Average_Grant_Aid,Total_Annual_Cost,Alumni_Salary,Acceptance_Rate,SAT_Lower,SAT_Upper,ACT_Lower,ACT_Upper
count,650.00000,650.000000,650.000000,648.000000,646.000000,650.000000,635.000000,648.000000,551.000000,551.000000,553.000000,553.000000
mean,325.50000,10002.692308,12022.290769,22336.947531,20031.208978,50330.175385,98851.968504,61.597222,1110.145191,1308.078040,23.278481,28.441230
std,187.78312,11162.172430,13175.091581,8269.912032,11175.869976,13223.055038,14335.249458,22.024781,127.744079,113.553237,3.729318,3.058115
min,1.00000,185.000000,386.000000,0.000000,2975.000000,0.000000,70700.000000,5.000000,820.000000,1060.000000,15.000000,20.000000
25%,163.25000,2020.250000,2240.750000,16410.000000,9288.250000,39917.000000,88600.000000,48.000000,1020.000000,1230.000000,21.000000,26.000000
50%,325.50000,4503.000000,6269.000000,21989.000000,19605.000000,50265.000000,96400.000000,67.000000,1080.000000,1290.000000,22.000000,28.000000
75%,487.75000,15657.000000,17788.000000,27580.500000,27474.500000,60771.750000,105600.000000,78.000000,1180.000000,1380.000000,25.000000,31.000000
max,650.00000,65100.000000,75044.000000,47270.000000,50897.000000,75735.000000,158200.000000,100.000000,1530.000000,1590.000000,34.000000,35.000000


---

## Analysis Questions

Now that the data is clean and you're familiar with its structure, work through the following
business-style questions. Each one mirrors the kind of question a real analyst might be asked about
this dataset.


### Question 1: Which Are the Top 5 Colleges in the Forbes Ranking?

**Business Context:** Before any deeper analysis, stakeholders usually want a quick answer:
who's at the very top?

**Concept:** This is **boolean/conditional filtering** — selecting rows of a DataFrame where a
condition on a column evaluates to `True`. Here the condition is based on the `Rank` column.

**Hint:** Try a condition like `ac.Rank <= 5` inside square brackets, or sort by `Rank` and use
`.head()`.

**Expected Output:** A DataFrame with 5 rows (all columns), showing the colleges ranked 1 through 5.


In [13]:
ac[ac.Rank <= 5]

,Rank,Name,City,State,Public_Private,Undergraduate_Population,Student_Population,Net_Price,Average_Grant_Aid,Total_Annual_Cost,Alumni_Salary,Acceptance_Rate,SAT_Lower,SAT_Upper,ACT_Lower,ACT_Upper,Website
0,1,Harvard University,Cambridge,MA,Private,13844,31120,14327.0,49870.0,69600,146800.0,5.0,1460.0,1590.0,32.0,35.0,www.harvard.edu
1,2,Stanford University,Stanford,CA,Private,8402,17534,13261.0,50134.0,69109,145200.0,5.0,1390.0,1540.0,32.0,35.0,www.stanford.edu
2,3,Yale University,New Haven,CT,Private,6483,12974,18627.0,50897.0,71290,138300.0,7.0,1460.0,1580.0,32.0,35.0,www.yale.edu
3,4,Massachusetts Institute of Technology,Cambridge,MA,Private,4680,11466,20771.0,43248.0,67430,155200.0,7.0,1490.0,1570.0,33.0,35.0,web.mit.edu
4,5,Princeton University,Princeton,NJ,Private,5659,8273,9327.0,48088.0,66150,139400.0,6.0,1430.0,1570.0,31.0,35.0,www.princeton.edu


### Question 2: Which State and City Do the Top 5 Colleges Belong To?

**Business Context:** Knowing location alongside rank helps identify regional clusters of
top-performing institutions.

**Concept:** **Column selection** — once you have a filtered/sliced set of rows, you can select
only specific columns of interest by passing a list of column names in double square brackets.

**Hint:** Combine your Question 1 approach with selecting only the `State`, `City`, and `Rank`
columns, e.g. `df[["State", "City", "Rank"]]`.

**Expected Output:** A DataFrame with 5 rows and exactly 3 columns: `State`, `City`, `Rank`.


In [14]:
ac[ac.Rank <= 5][["State", "City", "Rank"]]

,State,City,Rank
0,MA,Cambridge,1
1,CA,Stanford,2
2,CT,New Haven,3
3,MA,Cambridge,4
4,NJ,Princeton,5


### Question 3: How Many States (and Which Ones) Have Colleges Listed by Forbes?

**Business Context:** Understanding geographic spread tells us whether this ranking favors certain
regions of the country.

**Concept:** `.nunique()` counts the number of **unique** values in a column. `.groupby()` combined
with `.count()` groups rows by a category (here, `State`) and counts how many rows fall into each
group — useful for "how many per category" questions.

**Hint:** Use `ac.State.nunique()` for the count of unique states, and
`ac.groupby(by="State").count()` (selecting one column, e.g. `["Name"]`, to keep the output clean)
to see how many colleges per state.

**Expected Output:** A single integer for the unique state count, and a Series/table listing each
state alongside its number of colleges.


In [15]:
ac.State.nunique()

51

### Question 4: How Many Private vs. Public Colleges Are in the List?

**Business Context:** Public vs. private status often correlates with tuition cost and funding
source — a useful segmentation for prospective students comparing affordability.

**Concept:** `.value_counts()` is a quick way to count how many times each unique value appears in a
column — ideal for categorical columns with a small number of categories (like Public vs. Private).

**Hint:** Make sure you've cleaned the column name first (see Warm-Up 5), then call
`.value_counts()` on the `Public_Private` column.

**Expected Output:** A Series with two entries (`Public` and `Private`) and their respective counts,
which should sum to the total number of rows in the dataset.


In [16]:
ac.groupby("State")["Name"].count()

State
AK     2
AL     8
AR     5
AZ     4
CA    56
CO    12
CT    10
DC     5
DE     1
FL    20
GA    14
HI     1
IA    13
ID     4
IL    25
IN    19
KS     3
KY    12
LA     8
MA    31
MD    14
ME     6
MI    17
MN    19
MO    13
MS     5
MT     4
NC    19
ND     2
NE     6
NH     3
NJ    13
NM     4
NV     2
NY    68
OH    29
OK     6
OR    11
PA    40
RI     7
SC     9
SD     4
TN    14
TX    25
UT     5
VA    20
VT     5
WA    12
WI    13
WV     1
WY     1
Name: Name, dtype: int64

### Question 5: How Many Colleges Are There Within Each State? Which State Has the Most?

**Business Context:** This tells us which states are the most competitive/concentrated markets for
top-ranked colleges.

**Concept:** Building on `.groupby()` and `.count()` from Question 3, you can chain `.sort_values()`
to order the results, and use `.index[0]` to pull out just the top state's name.

**Hint:** Sort your grouped state counts in descending order with `.sort_values(ascending=False)`,
then use `.index[0]` to get the state with the highest count.

**Expected Output:** A sorted Series (state → count of colleges), and separately, a single string
value naming the state with the most colleges.


In [17]:
state_counts = ac.groupby("State")["Name"].count()
state_counts = state_counts.sort_values(ascending=False)

state_counts
state_counts.index[0]

'NY'

### Question 6: What Percentage of Students Are Undergraduates at Each College?

**Business Context:** A high undergraduate percentage may indicate a college focused primarily on
undergraduate education rather than graduate/research programs.

**Concept:** This is about **creating a new calculated column**. You can derive a new field from
existing ones using standard arithmetic operations directly on columns — pandas applies the
operation row-by-row automatically (this is called "vectorized" computation).

**Hint:** Divide the undergraduate population column by the total student population column,
multiply by 100, and round the result (e.g., using `.round(2)`). Assign this to a new column name.

**Expected Output:** The DataFrame should now have one additional column (e.g., `UnderGradPerc`)
containing percentage values, typically between 0 and 100.


In [18]:
ac["UnderGradPerc"] = (
    ac["Undergraduate_Population"] /
    ac["Student_Population"] * 100
).round(2)

### Question 7: Which 5 Colleges Have the Highest Percentage of Undergraduates?

**Business Context:** This narrows down Question 6 into an actionable "top 5" list.

**Concept:** `.sort_values(by=..., ascending=False)` orders rows by a specific column from highest
to lowest. Combine with `.head()` to get just the top N rows, and column selection to keep only the
relevant fields.

**Hint:** Sort by the new percentage column you created in Question 6, in descending order, then
take the top 5 rows and select just the `Name` and percentage columns.

**Expected Output:** A DataFrame with 5 rows and 2 columns: college name and undergraduate
percentage, sorted from highest to lowest.


In [19]:
ac.sort_values(
    by="UnderGradPerc",
    ascending=False
)[["Name", "UnderGradPerc"]].head()

,Name,UnderGradPerc
570,"University of Texas, Rio Grande Valley",99.99
565,Oakland University,99.94
253,Saint John's University (MN),99.94
364,Salisbury University,99.92
345,St. Norbert College,99.86


### Question 8: Which Colleges Receive the Highest Average Grant Aid?

**Business Context:** Grant aid (money that doesn't need to be repaid) is a major factor in
affordability — colleges offering the most aid may be more attractive to cost-conscious students.

**Concept:** Same technique as Question 7 — sorting and slicing — just applied to a different
column.

**Hint:** Sort by the `Average_Grant_Aid` column in descending order, then take the top rows and
select the `Name` and `Average_Grant_Aid` columns.

**Expected Output:** A DataFrame listing the top colleges by average grant aid, with college name
and grant aid amount as columns.


In [20]:
ac.sort_values(
    by="Average_Grant_Aid",
    ascending=False
)[["Name", "Average_Grant_Aid"]].head(10)

,Name,Average_Grant_Aid
2,Yale University,50897.0
1,Stanford University,50134.0
0,Harvard University,49870.0
27,Amherst College,49467.0
13,Columbia University,49185.0
11,Pomona College,48930.0
4,Princeton University,48088.0
18,Williams College,48050.0
8,Duke University,47836.0
9,Dartmouth College,47171.0


### Question 9: What Is the Average Undergraduate Population in the State of California ('CA')?

**Business Context:** Comparing a single state's average against the national picture is a common
"drill-down" style question.

**Concept:** This combines **row filtering** (restrict to one state) with an **aggregate function**
(`.mean()`) applied to a specific column.

**Hint:** Filter the DataFrame where `State` equals `"CA"`, select the undergraduate population
column, and call `.mean()` on it.

**Expected Output:** A single numeric value representing the average undergraduate population across
California colleges in the dataset.


In [21]:
ac[ac.State == "CA"]["Undergraduate_Population"].mean()

np.float64(13186.910714285714)

### Question 10: Which Colleges Have the Highest Chances of Admission?

**Business Context:** A high acceptance rate generally signals an easier college to get into —
useful for students weighing "reach" vs. "safety" school options.

**Concept:** Boolean filtering again, this time using a threshold condition on the
`Acceptance_Rate` column. Checking `.max()` first helps you understand the range before picking a
sensible threshold.

**Hint:** First check `ac.Acceptance_Rate.max()` to see the highest value in the dataset. Then
filter for colleges above a threshold you choose (e.g., greater than 60), and select the `Name` and
`Acceptance_Rate` columns.

**Expected Output:** A DataFrame listing colleges whose acceptance rate exceeds your chosen
threshold, with their name and acceptance rate.


In [22]:
ac.Acceptance_Rate.max()

100.0

In [23]:
ac[ac.Acceptance_Rate > 60][
    ["Name", "Acceptance_Rate"]
]

,Name,Acceptance_Rate
67,"University of Illinois, Urbana-Champaign",62.0
96,Yeshiva University,63.0
102,"Texas A&amp;M University, College Station",71.0
107,Virginia Tech,70.0
109,Rose-Hulman Institute of Technology,61.0
...,...,...
644,Morehouse College,74.0
645,New Mexico State University,64.0
646,Indiana State University,85.0
647,Emory &amp; Henry College,72.0


### Question 11: Are There Any 'Art' Colleges in the List?

**Business Context:** Sometimes you need to search for institutions matching a keyword in their
name, rather than filtering on a numeric threshold.

**Concept:** `.str.contains()` is a **string method** that checks whether a substring appears
anywhere within each value of a text column, returning `True`/`False` per row — which you can then
use as a filter condition.

**Hint:** Use `ac.Name.str.contains("Art")` as your filter condition inside square brackets.

**Expected Output:** A DataFrame containing only the rows where the college name includes the
substring "Art" (this may include colleges where "Art" appears as part of a longer word).


In [24]:
ac.Name.str.contains("Art")

0      False
1      False
2      False
3      False
4      False
       ...  
645    False
646    False
647    False
648    False
649    False
Name: Name, Length: 650, dtype: bool

In [25]:
ac[ac.Name.str.contains("Art")]

,Rank,Name,City,State,Public_Private,Undergraduate_Population,Student_Population,Net_Price,Average_Grant_Aid,Total_Annual_Cost,Alumni_Salary,Acceptance_Rate,SAT_Lower,SAT_Upper,ACT_Lower,ACT_Upper,Website,UnderGradPerc
302,303,Maryland Institute College of Art,Baltimore,MD,Private,2041,2128,38763.0,18939.0,62850,84500.0,62.0,1060.0,1290.0,24.0,29.0,NaN,95.91
401,402,California Institute of the Arts,Santa Clarita,CA,Private,991,1498,46101.0,15401.0,68993,91000.0,24.0,NaN,NaN,NaN,NaN,NaN,66.15


### Question 12: Divide Alumni Salary into Three Buckets (Low, Medium, High)

**Business Context:** Grouping a continuous variable like salary into broad tiers (Low/Medium/High)
makes it easier to summarize and compare across colleges, rather than working with exact figures.

**Concept:** `pd.cut()` **bins** a continuous numeric column into a specified number of equal-width
intervals, and can assign custom labels to each bin — turning a continuous variable into a
categorical one.

**Hint:** Use `pd.cut()` on the `Alumni_Salary` column, specifying `bins=3` and
`labels=["L", "M", "H"]`. Save the result into a new column. Then filter for rows where that new
column equals `"H"`.

**Expected Output:** A new categorical column with values `L`, `M`, or `H` for every row, and a
filtered DataFrame showing only the colleges that fall into the `"H"` (high salary) bucket.


In [26]:
ac["Salary_Bucket"] = pd.cut(
    ac["Alumni_Salary"],
    bins=3,
    labels=["L", "M", "H"]
)
ac["Salary_Bucket"].value_counts().sort_index()

Salary_Bucket
L    386
M    218
H     31
Name: count, dtype: int64

### Question 13: What Is the Mean Alumni Salary at Each Bucket Level?

**Business Context:** This validates that your bucketing in Question 12 makes sense — the "High"
bucket should indeed have a noticeably higher average salary than "Low."

**Concept:** `.groupby()` combined with `.mean()` computes the average of a numeric column, split
out by each category in another column — here, your salary bucket labels.

**Hint:** Group by your salary bin column and compute `.mean()` on the `Alumni_Salary` column.

**Expected Output:** A small table with 3 rows (one per bucket: L, M, H) showing the average alumni
salary for each group — values should increase from L to H.


In [27]:
ac.groupby(
    "Salary_Bucket",
    observed=False
)["Alumni_Salary"].mean()

Salary_Bucket
L     89881.088083
M    109127.981651
H    138290.322581
Name: Alumni_Salary, dtype: float64

### Question 14: Create a New DataFrame of Colleges with High Alumni Salary

**Business Context:** Once you've identified a subgroup of interest (colleges in the "High" alumni
salary bucket), it's common practice to save it as its own DataFrame for focused follow-up analysis.

**Concept:** This is simply **filtering and assignment** — the result of a boolean filter can be
stored in a new variable name, creating an independent DataFrame you can analyze further without
disturbing the original data.

**Hint:** Filter `ac` where your salary bucket column equals `"H"`, and assign the result to a new
variable name.

**Expected Output:** A new DataFrame containing only the "High" alumni salary colleges — check its
`.shape` to confirm how many colleges fall into this group.


In [28]:
high_salary_colleges = ac[
    ac["Salary_Bucket"] == "H"
]
high_salary_colleges.shape

(31, 19)

---

## Bonus: Indexing with `iloc` and `loc`

**Concept:** Pandas offers two primary ways to select subsets of a DataFrame by position or by
label:
- **`.iloc[]`** selects by **integer position** — e.g., "give me rows 2 and 3" regardless of their
  labels.
- **`.loc[]`** selects by **label** — e.g., "give me rows labeled 1 through 2" and "columns from
  `Public_Private` through `Alumni_Salary`" (note: with `.loc`, the end label is *included*, unlike
  standard Python slicing).

**Hint:** Try `ac.iloc[2:4, :]` to select rows by position, and
`ac.loc[1:2, "Public_Private":"Alumni_Salary"]` to select rows and a column range by label.

**Expected Output:** Two small DataFrames — one showing rows at positions 2 and 3 (all columns), and
one showing rows labeled 1 and 2, restricted to the column range specified.


In [29]:
ac.iloc[2:4, :]

,Rank,Name,City,State,Public_Private,Undergraduate_Population,Student_Population,Net_Price,Average_Grant_Aid,Total_Annual_Cost,Alumni_Salary,Acceptance_Rate,SAT_Lower,SAT_Upper,ACT_Lower,ACT_Upper,Website,UnderGradPerc,Salary_Bucket
2,3,Yale University,New Haven,CT,Private,6483,12974,18627.0,50897.0,71290,138300.0,7.0,1460.0,1580.0,32.0,35.0,www.yale.edu,49.97,H
3,4,Massachusetts Institute of Technology,Cambridge,MA,Private,4680,11466,20771.0,43248.0,67430,155200.0,7.0,1490.0,1570.0,33.0,35.0,web.mit.edu,40.82,H


In [30]:
ac.loc[1:2, "Public_Private":"Alumni_Salary"]

,Public_Private,Undergraduate_Population,Student_Population,Net_Price,Average_Grant_Aid,Total_Annual_Cost,Alumni_Salary
1,Private,8402,17534,13261.0,50134.0,69109,145200.0
2,Private,6483,12974,18627.0,50897.0,71290,138300.0


---

## Wrap-Up

You've now practiced the core EDA toolkit in pandas: inspecting structure, cleaning data, filtering,
grouping, sorting, deriving new columns, string search, binning, and positional/label-based
indexing. These same techniques apply directly to almost any tabular dataset you'll encounter —
whether it's college rankings, sales records, or loan applications.

**Happy Learning!**
